# Lecture 19: Sensitivity Analysis — Non-Linear Programming

---

```{note}
Lectures 16–18 each computed a KKT point — the optimal primal variables $x^*$ and the Lagrange multipliers $\lambda^*$ — using a different algorithm (Penalty, Barrier, Interior-Point). All three methods agreed on the same $\lambda^*$, but at the time we treated it only as a byproduct needed to certify optimality. In Lecture 8, we saw that for LP, the multipliers extracted from $\mathbf{B}^{-1}$ are far more than a certificate: they are shadow prices that tell a planner exactly how much an extra unit of a scarce resource is worth. The same is true here. This lecture shows that the KKT multiplier $\lambda_i^*$ is the NLP shadow price — and develops the resource ranging and cost ranging tools needed to know how far that price can be trusted.
```

**Learning Objectives**

By the end of this notebook, you will be able to:
1. Interpret the KKT multiplier $\lambda_i^*$ as the rate of change of the optimal objective value with respect to the $i$-th constraint's right-hand side (the NLP Sensitivity Theorem).
2. Determine the range of a constraint's right-hand side (resource ranging) and an objective coefficient (cost ranging) over which the current active set — and hence $\lambda^*$ — remains valid.
3. Distinguish primal feasibility and dual feasibility as the two distinct mechanisms that can bound a sensitivity range in NLP, and use both to make managerial recommendations in transportation contexts.

**Prerequisites**: NLP Principles (Lecture 12); Gradient Descent, Newton's Method, BFGS (Lectures 13–15); Penalty Method, Barrier Method, Interior-Point Method (Lectures 16–18); LP Sensitivity Analysis (Lecture 8); calculus (partial derivatives, implicit differentiation); basic linear algebra.

**Estimated time**: 90 minutes (including in-class exercises)

---

## Optimality Conditions — KKT

Consider the standard-form NLP:

$$\min_{x} \; f(x) \quad \text{subject to} \quad g_i(x) \leq 0, \quad i = 1, \ldots, m$$

where $f, g_i : \mathbb{R}^n \to \mathbb{R}$ are continuously differentiable. The **Lagrangian** is:

$$\mathcal{L}(x, \lambda) = f(x) + \sum_{i=1}^{m} \lambda_i \, g_i(x)$$

If $x^*$ is a local minimum and a constraint qualification holds (e.g., LICQ), then there exist multipliers $\lambda^* \in \mathbb{R}^m$ such that the following **Karush–Kuhn–Tucker (KKT) conditions** are satisfied:

| Condition | Statement | Interpretation |
|-----------|-----------|----------------|
| **Stationarity** | $\nabla f(x^*) + \sum_{i=1}^{m} \lambda_i^* \nabla g_i(x^*) = \mathbf{0}$ | Gradient of objective balanced by gradients of active constraints |
| **Primal feasibility** | $g_i(x^*) \leq 0 \quad \forall\, i$ | Solution lies within the feasible region |
| **Dual feasibility** | $\lambda_i^* \geq 0 \quad \forall\, i$ | Multipliers are non-negative for inequality constraints |
| **Complementary slackness** | $\lambda_i^* \, g_i(x^*) = 0 \quad \forall\, i$ | A constraint is either active ($g_i = 0$) or its multiplier is zero ($\lambda_i = 0$) |

For a **convex** NLP (convex $f$, convex $g_i$), the KKT conditions are both necessary and sufficient for global optimality. Lectures 16–18 each computed a point satisfying these four conditions; this lecture asks what else that point can tell us.

---

## The NLP Sensitivity Theorem

Write the right-hand side of constraint $i$ explicitly as a parameter $b_i$, so that $g_i(x) \leq 0$ becomes $\hat{g}_i(x) \leq b_i$, and let $f^*(\mathbf{b})$ denote the optimal objective value as a function of the resource vector $\mathbf{b}$. Provided the active set does not change and second-order sufficiency holds, the **Sensitivity Theorem** states:

$$\frac{\partial f^*}{\partial b_i} = -\lambda_i^*$$

This is the direct NLP analogue of the LP shadow price $\pi_i^* = \partial z^*/\partial b_i$ from Lecture 8 — with one sign convention to keep straight: because the standard-form constraint here is written $g_i(x) - b_i \leq 0$ rather than $\leq b_i$ directly as in the LP's $\mathbf{Ax} \leq \mathbf{b}$, relaxing the resource (increasing $b_i$) changes $f^*$ by $-\lambda_i^*$, not $+\lambda_i^*$. For a **minimization** problem with $\lambda_i^* \geq 0$, this means $f^*$ decreases as $b_i$ increases — exactly as intuition demands: more resource can only help, never hurt, the achievable minimum cost.

$$\boxed{\lambda_i^* = -\frac{\partial f^*}{\partial b_i} \;=\; \text{NLP shadow price of constraint } i}$$

| Quantity | LP (Lecture 8) | NLP (this lecture) |
|----------|-----------------|---------------------|
| Shadow price | $\pi_i^* = \mathbf{c}_B^\top \mathbf{B}^{-1}$, read off $\mathbf{B}^{-1}$ | $\lambda_i^*$, read off directly from the KKT point |
| Valid range | Piecewise-linear: exact via $\mathbf{B}^{-1}(\mathbf{b}+\Delta\mathbf{b}) \geq \mathbf{0}$ | Local: valid while active set & 2nd-order conditions hold |
| Behaviour outside range | Jumps to next vertex's $\pi_i^*$ (constant within range) | $f^*(\mathbf{b})$ may curve smoothly; $\lambda_i^*(\mathbf{b})$ can vary continuously |
| Source of the multiplier | Dual LP solution | Lagrange multiplier of the KKT system |

```{tip}
In LP, $z^*(\mathbf{b})$ is *piecewise linear* in $\mathbf{b}$, so the shadow price is a constant within each range. In NLP, $f^*(\mathbf{b})$ is generally a smooth nonlinear function of $\mathbf{b}$ — so $\lambda_i^*(\mathbf{b})$ may itself vary continuously with $b_i$ rather than staying constant. The "rate" interpretation of $\lambda_i^*$ is therefore strictly local: it is the *instantaneous* rate of change at the current $\mathbf{b}$, not a constant slope over the entire valid range.
```

We now make all of this concrete using the **NHAI parallel-links problem** from Lecture 18, whose KKT point we already verified by hand.

---

In [ ]:
# --- Imports and Setup ---
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.optimize import minimize

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12
print("Setup complete.")

## Worked Example: NHAI Parallel Links

*Same problem as Lecture 18, Exercise 2.* An NHAI corridor study models total system travel time across two parallel links — the Mumbai–Pune Expressway ($v_1$, thousand veh/hr) and old NH48 ($v_2$, thousand veh/hr):

$$\min_{v_1, v_2} \; T(v_1, v_2) = 3v_1^2 - 2v_1 v_2 + 2v_2^2 - 14v_1 - 12v_2 + 50$$

subject to a combined capacity constraint $v_1 + v_2 \leq b$, $v_1, v_2 \geq 0$, with baseline $b = 5$ (thousand veh/hr).

Lecture 18 verified by hand (Newton step on the primal-dual system) and confirmed numerically that the KKT point at $b = 5$ is:

$$v_1^* = \frac{16}{7}, \quad v_2^* = \frac{19}{7}, \quad \lambda^* = \frac{40}{7} \approx 5.714$$

with the capacity constraint active ($v_1^* + v_2^* = 5$) and both variables strictly positive (so neither non-negativity bound is active). We will now treat $b$ and the linear cost coefficient on $v_1$ as parameters and ask how far each can move before this KKT point — and the value of $\lambda^*$ — stops being valid.

---

In [ ]:
# --- Confirm the KKT point and the Sensitivity Theorem numerically ---

def T(v):
    v1, v2 = v
    return 3*v1**2 - 2*v1*v2 + 2*v2**2 - 14*v1 - 12*v2 + 50

def f_star(b):
    # Optimal T for capacity b, solved via SciPy.
    cons = ({'type': 'ineq', 'fun': lambda v: b - v[0] - v[1]})
    res = minimize(T, [1, 1], constraints=cons, bounds=[(0, None), (0, None)], tol=1e-12)
    return res.fun, res.x

f5, v5 = f_star(5.0)
print(f"At b = 5:  v* = ({v5[0]:.4f}, {v5[1]:.4f}),  T* = {f5:.4f}")
print(f"Hand-computed KKT point: v* = (16/7, 19/7) = ({16/7:.4f}, {19/7:.4f}),  lambda* = 40/7 = {40/7:.4f}")

# --- Sensitivity Theorem check: df*/db should equal -lambda* ---
h = 1e-3
df_db = (f_star(5 + h)[0] - f_star(5 - h)[0]) / (2*h)
print(f"\nFinite-difference dT*/db at b=5: {df_db:.4f}")
print(f"-lambda* = {-40/7:.4f}")
print(f"Match: {np.isclose(df_db, -40/7, atol=1e-3)}")

### Resource Ranging

Since the capacity constraint is active at the optimum, we can substitute $v_2 = b - v_1$ directly into $T$ and re-derive the unconstrained-in-$v_1$ minimum as a function of $b$:

$$T(v_1, b-v_1) = 7v_1^2 - 2v_1(3b+1) + 2b^2-12b+50$$

Setting $\partial T/\partial v_1 = 0$:

$$v_1^*(b) = \frac{3b+1}{7}, \qquad v_2^*(b) = b - v_1^*(b) = \frac{4b-1}{7}$$

Substituting back gives the optimal value function in closed form:

$$f^*(b) = \frac{5}{7}b^2 - \frac{90}{7}b + \frac{349}{7}$$

$$\lambda^*(b) = -\frac{df^*}{db} = \frac{90}{7} - \frac{10}{7}b$$

At $b=5$: $\lambda^*(5) = 90/7 - 50/7 = 40/7$ ✓ — matching the hand-computed value exactly.

**Validity range.** Two distinct mechanisms can break this formula, and both must be checked — this is the key NLP-specific lesson:

1. **Primal feasibility** of the active-set assumption: $v_1^*(b) \geq 0$ and $v_2^*(b) \geq 0$ must both hold, or the assumed active set (constraint binding, both bounds inactive) is wrong.
   $$v_1^*(b) \geq 0 \;\Rightarrow\; b \geq -\tfrac{1}{3} \qquad v_2^*(b) \geq 0 \;\Rightarrow\; b \geq \tfrac{1}{4}$$
2. **Dual feasibility**: $\lambda^*(b) \geq 0$, or the capacity constraint is no longer binding at all (the unconstrained optimum becomes feasible).
   $$\lambda^*(b) \geq 0 \;\Rightarrow\; \tfrac{90}{7} - \tfrac{10}{7}b \geq 0 \;\Rightarrow\; b \leq 9$$

The binding lower bound is $v_2^*(b) \geq 0 \Rightarrow b \geq 1/4$, and the binding upper bound is $\lambda^*(b) \geq 0 \Rightarrow b \leq 9$. So:

$$\boxed{b \in \left[\tfrac{1}{4},\ 9\right]}$$

Within this range, $\lambda^*(b) = \frac{90}{7}-\frac{10}{7}b$ gives the *exact, locally-instantaneous* value of relaxing capacity by one unit — but note that, unlike LP, this rate itself changes continuously with $b$ rather than staying constant.

> **Managerial interpretation**: As long as combined corridor capacity stays between 0.25 and 9 thousand veh/hr, the optimum continues to split flow between both links with the capacity constraint binding. At $b=9$, the constraint stops binding altogether — NHAI would have enough combined capacity that travel time is minimized at the unconstrained optimum $(v_1,v_2)=(4,5)$, and adding further capacity yields zero marginal benefit ($\lambda^*=0$).


In [ ]:
# --- Resource ranging: f*(b) and lambda*(b) over their valid range ---

b_vals = np.linspace(0.25, 9, 200)
f_star_vals = (5/7)*b_vals**2 - (90/7)*b_vals + 349/7
lambda_vals = (90/7) - (10/7)*b_vals

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(b_vals, f_star_vals, color='tab:blue', lw=2)
ax1.axvline(5, color='gray', ls='--', lw=1, label=r'baseline $b=5$')
f_star_at_5 = (5/7)*5**2 - (90/7)*5 + 349/7
ax1.scatter([5], [f_star_at_5], color='tab:red', zorder=5, label=r'$T^*(5)$')
ax1.set_xlabel(r'Capacity $b$ (thousand veh/hr)')
ax1.set_ylabel(r'$T^*(b)$')
ax1.set_title(r'Optimal value function $T^*(b)$')
ax1.legend()

ax2.plot(b_vals, lambda_vals, color='tab:orange', lw=2)
ax2.axvline(5, color='gray', ls='--', lw=1, label=r'baseline $b=5$')
ax2.axhline(0, color='black', lw=0.8)
ax2.scatter([5], [40/7], color='tab:red', zorder=5, label=r'$\lambda^*(5)=40/7$')
ax2.set_xlabel(r'Capacity $b$ (thousand veh/hr)')
ax2.set_ylabel(r'$\lambda^*(b)$')
ax2.set_title(r'Shadow price $\lambda^*(b)$')
ax2.legend()

plt.tight_layout()
plt.show()

print("Note: lambda*(b) is LINEAR in b here (not constant, as it would be in LP) -- ")
print("this is because the active-set structure stays fixed across the whole [1/4, 9] range,")
print("but the underlying objective is quadratic rather than linear.")

### Cost Ranging

Now hold $b=5$ fixed and ask how far the linear cost coefficient on $v_1$ (currently $-14$) can shift before the active set changes. Write the perturbed objective as:

$$T(v_1, v_2; \Delta c) = 3v_1^2 - 2v_1v_2 + 2v_2^2 + (-14+\Delta c)v_1 - 12v_2 + 50$$

Substituting the active constraint $v_2 = 5 - v_1$ and re-solving $\partial T/\partial v_1 = 0$:

$$v_1^*(\Delta c) = \frac{16}{7} - \frac{\Delta c}{14}, \qquad v_2^*(\Delta c) = \frac{19}{7} + \frac{\Delta c}{14}$$

From stationarity, $\lambda(\Delta c) = -\bigl[6v_1^*(\Delta c) - 2v_2^*(\Delta c) + (-14+\Delta c)\bigr]$, which simplifies to:

$$\lambda(\Delta c) = \frac{40}{7} - \frac{3\Delta c}{7}$$

Checking both feasibility mechanisms again:
- Primal feasibility ($v_1^*\geq 0$, $v_2^*\geq 0$): $v_1^*(\Delta c)\geq 0 \Rightarrow \Delta c \leq 32$; $v_2^*(\Delta c)\geq 0 \Rightarrow \Delta c \geq -38$.
- Dual feasibility ($\lambda(\Delta c) \geq 0$): $\Delta c \leq \tfrac{40}{3} \approx 13.33$.

The binding bound is dual feasibility, **not** primal feasibility:

$$\boxed{\Delta c \in \left[-38,\ \tfrac{40}{3}\right]}, \qquad \text{i.e. the coefficient on } v_1 \in \left[-52,\ -14+\tfrac{40}{3}\right] = \left[-52,\ -\tfrac{2}{3}\right]$$

```{caution}
A student instinctively checks only $v_1^*, v_2^* \geq 0$ and would report $\Delta c \in [-38, 32]$ — too wide. The active set does change at $\Delta c = 40/3$, well before $v_1$ or $v_2$ would have gone negative: at that point $\lambda$ hits zero and the capacity constraint stops binding altogether. **Always check dual feasibility ($\lambda \geq 0$) alongside primal feasibility when ranging in NLP** — in LP this distinction is invisible because the simplex tableau already enforces both simultaneously, but in NLP the two checks must be made separately.
```

> **Managerial interpretation**: The current flow split $(v_1^*,v_2^*)=(16/7,19/7)$ and the value $\lambda^*=40/7$ remain valid as long as the marginal cost coefficient on the expressway stays between $-52$ and $-2/3$. Beyond $-2/3$ (i.e. the expressway becomes cheap enough, in relative terms, that it's no longer worth restricting), the capacity constraint stops binding and the corridor relaxes to its unconstrained split.

---

In [ ]:
# --- Cost ranging: verify by re-solving NLP at perturbed cost coefficients ---

def T_perturbed(v, dc):
    v1, v2 = v
    return 3*v1**2 - 2*v1*v2 + 2*v2**2 + (-14+dc)*v1 - 12*v2 + 50

def solve_perturbed(dc):
    cons = ({'type': 'ineq', 'fun': lambda v: 5 - v[0] - v[1]})
    res = minimize(T_perturbed, [1, 1], args=(dc,), constraints=cons,
                    bounds=[(0, None), (0, None)], tol=1e-12)
    return res.x

for dc in [-38, -20, 0, 13.0, 40/3, 14, 20]:
    v1_opt, v2_opt = solve_perturbed(dc)
    active = (v1_opt + v2_opt) > 4.999
    print(f"dc={dc:7.3f}:  v1*={v1_opt:.4f}, v2*={v2_opt:.4f}, "
          f"sum={v1_opt+v2_opt:.4f}, constraint active: {active}")

print(f"\nPredicted breakpoint: dc = 40/3 = {40/3:.4f}")
print("Above this, the constraint stops binding -- matches the dual-feasibility bound, not primal.")

## Key Insights

1. **The multiplier already is the shadow price.** Lectures 16–18 spent considerable effort computing $\lambda^*$ as a *certificate* of optimality. This lecture shows it was never just a certificate — by the Sensitivity Theorem, $\lambda^*$ is directly the rate of change of the optimal cost with respect to the corresponding resource. No extra computation (no $\mathbf{B}^{-1}$, no re-solving) is needed to get the rate; only to find the *range* over which that rate stays valid.

2. **Two distinct feasibility checks, not one.** In LP, ranging via $\mathbf{B}^{-1}(\mathbf{b}+\Delta\mathbf{b}) \geq \mathbf{0}$ automatically captures both primal and dual breakdowns because the simplex basis encodes both. In NLP, primal feasibility ($x^*(\theta) \geq \mathbf{0}$ or within bounds) and dual feasibility ($\lambda^*(\theta) \geq 0$) must be checked **separately**, and either can be the binding constraint — the NHAI cost-ranging example shows dual feasibility binding first, well inside the region where primal feasibility would still hold.

3. **The shadow price is a local, not a global, constant.** Because NLP objectives are typically nonlinear, $\lambda^*(\theta)$ itself usually varies continuously across its valid range — unlike the LP case, where $\pi_i^*$ is a flat constant between breakpoints. A manager asking "what's it worth?" gets a number that is exactly correct at the current operating point but degrades gracefully (not discretely) as conditions move away from it.

## ⚠️ Common Pitfalls

| ✗ Wrong | ✓ Correct |
|---------|-----------|
| Checking only $x^*(\theta) \geq \mathbf{0}$ to find the ranging interval | Check **both** primal feasibility ($x^*(\theta)\geq\mathbf{0}$) and dual feasibility ($\lambda^*(\theta)\geq 0$); report the tighter of the two bounds |
| Treating $\lambda^*$ as constant across the whole valid range, the way an LP shadow price is | Recompute or re-derive $\lambda^*(\theta)$ at the new operating point if $\theta$ has moved meaningfully — the rate itself may have changed |
| Applying the Sensitivity Theorem at a point where the active set is uncertain or the problem is non-convex | Verify the active set is stable and second-order sufficiency holds before trusting $\partial f^*/\partial b_i = -\lambda_i^*$ |

---

## In-Class Exercises

### Exercise 1

*Same problem as Lecture 18, In-Class Exercise 1.* MTC Chennai is optimizing a demand-responsive feeder service on the Tambaram–Chromepet corridor. The daily operating cost depends on fleet size $n$ (vehicles) and scheduled headway $h$ (minutes):

$$C(n, h) = (n - 5)^2 + 2(h - 3)^2$$

subject to a depot resource constraint $n + h \leq b$, with baseline $b = 7$. Lecture 18 verified by hand that the KKT point at $b=7$ is $n^*=13/3$, $h^*=8/3$, $\lambda^*=4/3$.

1. Substitute the active constraint $h = b - n$ into $C$, and re-derive $n^*(b)$, $h^*(b)$, and $C^*(b)$ as closed-form functions of $b$. Confirm $C^*(7)$ matches the Lecture 18 value.
2. Differentiate to obtain $\lambda^*(b) = -dC^*/db$, and confirm $\lambda^*(7) = 4/3$.
3. Find the resource-ranging interval for $b$, checking **both** primal feasibility ($n^*(b)\geq 0$, $h^*(b)\geq 0$) and dual feasibility ($\lambda^*(b) \geq 0$). Which bound binds on each side?
4. Now hold $b=7$ fixed and let the fleet-size target shift: replace $(n-5)^2$ with $(n-5-\Delta c)^2$. Re-derive $n^*(\Delta c)$, $h^*(\Delta c)$, and $\lambda(\Delta c)$. Find the valid range of $\Delta c$.
5. The MTC depot manager asks: "If I free up one more unit of combined depot capacity, exactly how much does today's operating cost fall?" Answer using $\lambda^*(7)$, then verify numerically using `scipy.optimize.minimize` at $b=7$ and $b=8$.

### Exercise 2

*Same problem as Lecture 18, Take-Away Exercise 1.* Delhivery is minimizing the daily operating cost at its Chennai cross-dock facility, with cost depending on daily throughput $p$ (tonnes/day) and staffing level $w$ (workers):

$$C(p, w) = 2p^2 - 4pw + 3w^2 - 12p - 6w + 50$$

subject to a combined budget-shift constraint $p + 2w \leq b$, baseline $b=14$. The Lecture 18 KKT point is $p^*=116/19$, $w^*=75/19$, $\lambda^*=64/19$.

1. Substitute the active constraint $w = (b-p)/2$ into $C$, and derive $p^*(b)$ and $C^*(b)$ in closed form.
2. Derive $\lambda^*(b)$ and confirm it matches $64/19$ at $b=14$.
3. Find the resource-ranging interval for $b$. (*Hint: as $b$ decreases, which variable hits zero first — $p^*$ or $w^*$?*)
4. The Delhivery facility manager originally asked (Lecture 18) what happens if the budget increases from 14 to 15. Answer this precisely now using $\lambda^*(b)$ rather than the single-point estimate from Lecture 18, and explain why the two answers differ slightly.

---

## Take-Away Exercises

For each problem below: derive $f^*(\theta)$ in closed form along the active constraint, derive $\lambda^*(\theta) = -df^*/d\theta$, find the resource- and cost-ranging intervals (checking both primal and dual feasibility), and state the managerial interpretation.

### Exercise 1

*Same problem as Lecture 18, Take-Away Exercise 2.* BMTC is optimizing its demand-responsive feeder service in Bengaluru. The total daily operating cost depends on fleet size $n$ (vehicles) and headway $h$ (minutes):

$$C(n, h) = 2n^2 + 3h^2 + 2nh - 20n - 30h + 100$$

subject to a depot capacity constraint $n \leq b$, baseline $b=2$. The Lecture 18 KKT point is $n^*=2$, $h^*=13/3$, $\nu^*=10/3$.

1. For fixed $n=b$, minimize over $h$ to obtain $h^*(b)$, then derive $C^*(b)$ and $\nu^*(b) = -dC^*/db$ in closed form. Confirm $\nu^*(2) = 10/3$.
2. Find the resource-ranging interval for $b$. (*Note: $h$ is unconstrained here, so only the constraint $n \leq b$ and dual feasibility $\nu \geq 0$ can bind — there is no lower primal-feasibility bound on $n$ from this constraint alone. Does the problem have an implicit $n\geq 0$ bound to consider?*)
3. BMTC's depot engineer reports the depot can be expanded to allow $n \leq 3$. Using $\nu^*(b)$, predict the new operating cost, then verify by re-solving the NLP numerically.

### Exercise 2

A Kochi container freight station is minimizing daily handling cost as a function of throughput at two berths, $x_1$ and $x_2$ (TEU/hr):

$$C(x_1, x_2) = 4x_1^2 - 2x_1x_2 + 3x_2^2 - 20x_1 - 16x_2 + 80$$

subject to a combined crane-hour constraint $x_1 + x_2 \leq b$, baseline $b = 6$, and $x_1, x_2 \geq 0$.

1. Find the KKT point at $b=6$ by hand: substitute the active constraint, solve the resulting one-variable stationarity condition, and recover $\lambda^*$.
2. Derive $C^*(b)$ and $\lambda^*(b)$ in closed form, and find the resource-ranging interval for $b$.
3. Now fix $b=6$ and perturb the linear coefficient on $x_1$ (currently $-20$) by $\Delta c$. Derive $\lambda(\Delta c)$ and find the cost-ranging interval. Which feasibility mechanism binds?
4. Implement a Python function `sensitivity_report(b)` that takes a capacity value, solves the NLP via `scipy.optimize.minimize`, and returns $(x_1^*, x_2^*, C^*, \hat\lambda)$, where $\hat\lambda$ is recovered numerically as the constraint's Lagrange multiplier (`res.get('multipliers')` or via the KKT residual, as in Lecture 18). Compare $\hat\lambda$ against your closed-form $\lambda^*(b)$ across $b \in \{4, 5, 6, 7, 8\}$.

---

## Circling Back

- **Lecture 8 (LP Sensitivity Analysis)**: The structure of this lecture mirrors Lecture 8 closely — shadow prices, resource ranging, and cost ranging all reappear. The key difference is *where the multiplier comes from* (the simplex basis $\mathbf{B}^{-1}$ in LP vs. the KKT stationarity condition in NLP) and *how ranging is computed* (closed-form vertex enumeration in LP vs. local active-set/second-order analysis in NLP).
- **Lectures 16–18 (Penalty, Barrier, Interior-Point Methods)**: Each method converges to the same KKT point and the same $\lambda^*$ — this lecture is the natural sequel, extracting managerial value from that shared output rather than treating $\lambda^*$ as a stopping-criterion artefact.
- **Lecture 12 (NLP Principles)**: The Taxonomy table previewed that "Lagrange multipliers measure constraint sensitivity" well before Lectures 16–18 introduced the machinery to compute them — this lecture is where that early preview is finally made precise and operational.

## Moving Forward

- **Lecture 20 (NLP Duality)**: The Sensitivity Theorem used here ($\lambda_i^* = -\partial f^*/\partial b_i$) is a special case of a much more general result connecting the primal NLP to its **dual problem** — exactly as LP shadow prices in Lecture 9 turned out to be the optimal dual variables. NLP Duality formalizes this connection and shows when strong duality (zero optimality gap) holds.
- **Lecture 21 (TE Problems III — Facility Location)**: Facility location problems will use sensitivity analysis to answer questions like "how much would relaxing a demand-coverage requirement save?" — directly applying the resource-ranging machinery developed here.

---

## Further Reading

- Nocedal, J. and Wright, S.J. (2006). *Numerical Optimization* (2nd ed.). Springer — Chapter 12 (theory of constrained optimization, sensitivity and active-set behaviour).
- Bazaraa, M.S., Sherali, H.D., and Shetty, C.M. (2006). *Nonlinear Programming: Theory and Algorithms* (3rd ed.). Wiley — Chapter 4 (Lagrangian duality and sensitivity for nonlinear programs).
- Fiacco, A.V. (1983). *Introduction to Sensitivity and Stability Analysis in Nonlinear Programming*. Academic Press — the canonical reference on parametric NLP sensitivity.
- SciPy documentation: `scipy.optimize.minimize` with `method='SLSQP'` or `'trust-constr'` returns Lagrange multiplier estimates via `res.v` / KKT residuals — [docs.scipy.org](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)